<a href="https://colab.research.google.com/github/NikhilGeorge01/Cross-Lingual-Voice-Cloning/blob/main/clean_VCT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q qwen-tts soundfile openai-whisper deep-translator


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.7 MB/s eta 0:00:00


In [ ]:
# Reinstall torch with CUDA support for T4
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU!")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [ ]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS Base model for voice cloning...")
print("Downloading ~4.5GB on first run — please wait...\n")

# Base model is the one that supports real voice cloning from audio
tts_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,   # bfloat16 works well on T4
)

print("✅ Model loaded!")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
Loading Qwen3-TTS Base model for voice cloning...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Model loaded!
VRAM used: 4.20 GB


In [ ]:
import whisper
whisper_model = whisper.load_model("large")

100%|█████████████████████████████████████| 2.88G/2.88G [00:40<00:00, 75.5MiB/s]


In [ ]:
from google.colab import files
from IPython.display import Audio, display
import subprocess

print("🎙️ Upload your ENGLISH reference recording (9-15 seconds of clear speech)...")
uploaded = files.upload()
ref_filename = list(uploaded.keys())[0]

# Convert to WAV
subprocess.run(["ffmpeg", "-y", "-i", ref_filename, "-ar", "16000", "-ac", "1", "reference_english.wav"],
               capture_output=True)

# The exact words spoken in your reference recording
REF_TEXT = "The quick brown fox jumps over the lazy dog near the riverbank every single morning."
# ⬆️ IMPORTANT: change this to match what YOU actually said in your recording

print("✅ Reference voice saved")
display(Audio("reference_english.wav"))

🎙️ Upload your ENGLISH reference recording (9-15 seconds of clear speech)...


Saving Record (online-voice-recorder.com) (1).mp3 to Record (online-voice-recorder.com) (1).mp3
✅ Reference voice saved


In [ ]:
# Set this to match your audio: "te" = Telugu, "ml" = Malayalam, "hi" = Hindi
SOURCE_LANG = "te"  # ← change this before running

lang_names = {"te": "Telugu", "ml": "Malayalam", "hi": "Hindi"}
print(f"🎙️ Upload your {lang_names[SOURCE_LANG]} audio recording...")
uploaded = files.upload()
source_filename = list(uploaded.keys())[0]

subprocess.run(["ffmpeg", "-y", "-i", source_filename, "-ar", "16000", "-ac", "1", "input_audio.wav"],
               capture_output=True)

print(f"✅ {lang_names[SOURCE_LANG]} audio saved")
display(Audio("input_audio.wav"))

🎙️ Upload your Telugu audio recording...


Saving Recording (13).m4a to Recording (13).m4a
✅ Telugu audio saved


In [ ]:
from deep_translator import GoogleTranslator

# Step 1: Transcribe
print(f"📝 Transcribing {lang_names[SOURCE_LANG]}...")
result = whisper_model.transcribe(
    "input_audio.wav",
    language=SOURCE_LANG,
    task="transcribe",
    temperature=0.0,
    best_of=5,
    beam_size=5,
)
source_text = result["text"].strip()
print(f"   {lang_names[SOURCE_LANG]}: {source_text}")

# Step 2: Translate to English
print("\n🌐 Translating to English...")
english_text = GoogleTranslator(source=SOURCE_LANG, target="en").translate(source_text)
print(f"   English : {english_text}")

# Step 3: Synthesize in cloned voice
print("\n🔊 Synthesizing in your cloned voice...")
wavs, sr = tts_model.generate_voice_clone(
    text=english_text,
    ref_audio="reference_english.wav",
    ref_text=REF_TEXT,
    language="English",
)
sf.write("final_output.wav", wavs[0], sr)
print("✅ Done! Saved to final_output.wav")

📝 Transcribing Telugu...
   Telugu: నా పేరు నికిల్ జార్జ్ నేను హాఇదురాబాద్ రో ఉంటాము

🌐 Translating to English...
   English : My name is Nikhil George and I live in Hyderabad Road

🔊 Synthesizing in your cloned voice...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Done! Saved to final_output.wav


In [ ]:
display(Audio("final_output.wav"))

In [ ]:
!pip install -q jiwer sacrebleu resemblyzer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 92.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import soundfile as sf
from jiwer import wer, cer
from sacrebleu.metrics import BLEU, CHRF
from resemblyzer import VoiceEncoder, preprocess_wav
from pathlib import Path

# ─── 1. TRANSLATION QUALITY ───────────────────────────────────────────────────
# Back-translate the output English → source language, then compare with original
print("=" * 55)
print("📊  TRANSLATION METRICS")
print("=" * 55)

# Back-translate English → source language for round-trip check
back_translated = GoogleTranslator(source="en", target=SOURCE_LANG).translate(english_text)
print(f"  Original  ({lang_names[SOURCE_LANG]}): {source_text}")
print(f"  English               : {english_text}")
print(f"  Back-translated       : {back_translated}")

# ChrF score — works on character level, good for non-English
chrf = CHRF()
chrf_score = chrf.sentence_score(back_translated, [source_text])
print(f"\n  ChrF score (0–100, higher=better) : {chrf_score.score:.1f}")

# Word Error Rate between original and back-translation (lower=better)
try:
    wer_score = wer(source_text, back_translated)
    print(f"  Round-trip WER  (0–1, lower=better): {wer_score:.3f}")
except:
    print("  Round-trip WER : could not compute (likely script mismatch)")

# ─── 2. VOICE SIMILARITY ──────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("🎙️  VOICE SIMILARITY METRICS")
print("=" * 55)

encoder = VoiceEncoder()

ref_wav   = preprocess_wav(Path("reference_english.wav"))
clone_wav = preprocess_wav(Path("final_output.wav"))

ref_embed   = encoder.embed_utterance(ref_wav)
clone_embed = encoder.embed_utterance(clone_wav)

# Cosine similarity (1.0 = identical voice)
cosine_sim = np.dot(ref_embed, clone_embed) / (
    np.linalg.norm(ref_embed) * np.linalg.norm(clone_embed)
)
print(f"  Cosine similarity (0–1, higher=better): {cosine_sim:.4f}")

# Rough interpretation
if cosine_sim > 0.85:
    verdict = "🟢 Excellent — very close to reference voice"
elif cosine_sim > 0.70:
    verdict = "🟡 Good — recognisably similar"
elif cosine_sim > 0.55:
    verdict = "🟠 Fair — some resemblance"
else:
    verdict = "🔴 Poor — voice drifted significantly"
print(f"  Verdict: {verdict}")

# ─── 3. AUDIO SANITY CHECKS ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print("🔊  AUDIO SANITY")
print("=" * 55)

for label, path in [("Reference", "reference_english.wav"), ("Output", "final_output.wav")]:
    audio, sr = sf.read(path)
    duration   = len(audio) / sr
    rms        = np.sqrt(np.mean(audio**2))
    peak       = np.max(np.abs(audio))
    print(f"  {label:10s} | Duration: {duration:.1f}s | RMS: {rms:.4f} | Peak: {peak:.4f}")

print("\n✅ Evaluation complete")

📊  TRANSLATION METRICS
  Original  (Telugu): నా పేరు నికిల్ జార్జ్ నేను హాఇదురాబాద్ రో ఉంటాము
  English               : My name is Nikhil George and I live in Hyderabad Road
  Back-translated       : నా పేరు నిఖిల్ జార్జ్ మరియు నేను హైదరాబాద్ రోడ్‌లో నివసిస్తున్నాను

  ChrF score (0–100, higher=better) : 49.4
  Round-trip WER  (0–1, lower=better): 0.625

🎙️  VOICE SIMILARITY METRICS
Loaded the voice encoder model on cuda in 0.09 seconds.
  Cosine similarity (0–1, higher=better): 0.9206
  Verdict: 🟢 Excellent — very close to reference voice

🔊  AUDIO SANITY
  Reference  | Duration: 9.8s | RMS: 0.0288 | Peak: 0.3860
  Output     | Duration: 3.9s | RMS: 0.0305 | Peak: 0.2041

✅ Evaluation complete


In [ ]:
"""
# ================================
# 0. MOUNT GOOGLE DRIVE (NEW)
# ================================
from google.colab import drive
drive.mount('/content/drive')

# ================================
# 1. SETUP
# ================================
import os
import zipfile
import requests
import torch
import torchaudio
import numpy as np
import random

from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ✅ NEW: Save directory in Drive
SAVE_DIR = "/content/drive/MyDrive/emotion_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================================
# 2. DOWNLOAD RAVDESS
# ================================
RAVDESS_URL = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip"
RAVDESS_ZIP = "ravdess.zip"
RAVDESS_DIR = "ravdess"

if not os.path.exists(RAVDESS_DIR):
    print("⬇️ Downloading RAVDESS...")
    r = requests.get(RAVDESS_URL, stream=True)
    with open(RAVDESS_ZIP, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)

    print("📦 Extracting...")
    with zipfile.ZipFile(RAVDESS_ZIP, "r") as z:
        z.extractall(RAVDESS_DIR)
    os.remove(RAVDESS_ZIP)
    print("✅ Done")
else:
    print("✅ Dataset already exists")

# ================================
# 3. LABEL MAP
# ================================
EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgusted",
    "08": "surprised",
}

LABEL2ID = {k: i for i, k in enumerate(EMOTION_MAP.values())}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ================================
# 4. LOAD DATASET
# ================================
samples = []

for wav_path in Path(RAVDESS_DIR).rglob("*.wav"):
    parts = wav_path.stem.split("-")
    if len(parts) < 3:
        continue
    emotion_code = parts[2]
    label_str = EMOTION_MAP.get(emotion_code)
    if label_str is None:
        continue

    samples.append({
        "path": str(wav_path),
        "label": LABEL2ID[label_str]
    })

print("\n📊 Dataset:")
counts = Counter(ID2LABEL[s["label"]] for s in samples)
for k, v in counts.items():
    print(f"{k:12s}: {v}")
print("TOTAL:", len(samples))

if len(samples) == 0:
    raise ValueError("Dataset not found properly!")

# ================================
# 5. SPLIT
# ================================
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.15,
    random_state=42,
    stratify=[s["label"] for s in samples]
)

print(f"\nTrain: {len(train_samples)} | Val: {len(val_samples)}")

# ================================
# 6. DATASET CLASS
# ================================
class EmotionDataset(Dataset):
    def __init__(self, samples, processor, augment=False):
        self.samples = samples
        self.processor = processor
        self.augment = augment
        self.max_len = 16000 * 4

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        wav, sr = torchaudio.load(s["path"])
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)

        wav = wav.mean(dim=0)

        if self.augment:
            if random.random() < 0.5:
                wav = wav + torch.randn_like(wav) * 0.003

        if len(wav) < self.max_len:
            wav = torch.nn.functional.pad(wav, (0, self.max_len - len(wav)))
        else:
            wav = wav[:self.max_len]

        inputs = self.processor(
            wav.numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "label": torch.tensor(s["label"])
        }

# ================================
# 7. MODEL
# ================================
MODEL_NAME = "facebook/wav2vec2-base"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=8,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model.to(DEVICE)

# ================================
# 8. LOADERS
# ================================
train_dataset = EmotionDataset(train_samples, processor, augment=True)
val_dataset = EmotionDataset(val_samples, processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# ================================
# 9. TRAINING
# ================================
optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
best_val = 0
patience = 3
counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_correct = 0
    total = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_values = batch["input_values"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        preds = outputs.logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        total += len(labels)

    scheduler.step()
    train_acc = train_correct / total

    # ---- validation ----
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in val_loader:
            input_values = batch["input_values"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(input_values=input_values)
            preds = outputs.logits.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += len(labels)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train={train_acc:.3f} | Val={val_acc:.3f}")

    # ================================
    # ✅ FIXED SAVING (IMPORTANT)
    # ================================
    if val_acc > best_val:
        best_val = val_acc
        counter = 0

        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)

        print(f"💾 Model saved to {SAVE_DIR}")

    else:
        counter += 1
        if counter >= patience:
            print("⏹ Early stopping")
            break

# ================================
# 10. OVERFITTING CHECK
# ================================
print("\n📊 Overfitting Check:")

gap = train_acc - best_val

print(f"Train Acc : {train_acc:.3f}")
print(f"Val Acc   : {best_val:.3f}")
print(f"Gap       : {gap:.3f}")

if gap < 0.05:
    print("🟢 No overfitting")
elif gap < 0.15:
    print("🟡 Mild overfitting")
else:
    print("🔴 Strong overfitting")


"""

'\n# ================================\n# 0. MOUNT GOOGLE DRIVE (NEW)\n# ================================\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\n# ================================\n# 1. SETUP\n# ================================\nimport os\nimport zipfile\nimport requests\nimport torch\nimport torchaudio\nimport numpy as np\nimport random\n\nfrom pathlib import Path\nfrom collections import Counter\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification\nfrom torch.optim import AdamW\nfrom torch.optim.lr_scheduler import CosineAnnealingLR\nfrom sklearn.model_selection import train_test_split\n\nrandom.seed(42)\ntorch.manual_seed(42)\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nprint("Using device:", DEVICE)\n\n# ✅ NEW: Save directory in Drive\nSAVE_DIR = "/content/drive/MyDrive/emotion_model"\nos.makedirs(SAVE_DIR, exist_ok=True)\n\n# ================================\n# 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

LOAD_DIR = "/content/drive/MyDrive/emotion_model"
print(os.listdir(LOAD_DIR))

['config.json', 'model.safetensors', 'processor_config.json', 'vocab.json', 'tokenizer_config.json']


In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LOAD_DIR = "/content/drive/MyDrive/emotion_model"

processor = Wav2Vec2Processor.from_pretrained(LOAD_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(LOAD_DIR)

model.to(DEVICE)
model.eval()

print("✅ MODEL LOADED SUCCESSFULLY")

✅ MODEL LOADED SUCCESSFULLY


In [ ]:
ID2LABEL = {
    0: "neutral",
    1: "calm",
    2: "happy",
    3: "sad",
    4: "angry",
    5: "fearful",
    6: "disgusted",
    7: "surprised",
}

In [ ]:
'''
from pathlib import Path
import shutil
import random
from IPython.display import Audio, display  # <-- added

# find all wav files
all_wavs = list(Path("ravdess").rglob("*.wav"))

# pick random file
sample_path = random.choice(all_wavs)

# copy as sample.wav
shutil.copy(sample_path, "sample.wav")

print("✅ Sample saved as sample.wav")
print("Original file:", sample_path)

# 🔊 play audio
display(Audio("sample.wav"))  # <-- added
'''

IndexError: Cannot choose from an empty sequence

In [ ]:
import torchaudio

def predict(audio_path):
    wav, sr = torchaudio.load(audio_path)

    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    wav = wav.mean(dim=0)

    inputs = processor(
        wav.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(inputs.input_values.to(DEVICE)).logits

    pred = logits.argmax(dim=1).item()
    return ID2LABEL[pred]

print(predict("/content/reference_english.wav"))

sad


In [ ]:
'''
# ==========================================
# 🔥 SMALL UNSEEN TEST SET (RAVDESS)
# ==========================================

import torch
import torchaudio
import numpy as np
import random
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ================================
# LABEL MAP
# ================================
LABEL2ID = {
    "neutral": 0,
    "calm": 1,
    "happy": 2,
    "sad": 3,
    "angry": 4,
    "fearful": 5,
    "disgusted": 6,
    "surprised": 7,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgusted",
    "08": "surprised",
}

# ================================
# LOAD MODEL
# ================================
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

LOAD_DIR = "/content/drive/MyDrive/emotion_model"

processor = Wav2Vec2Processor.from_pretrained(LOAD_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(LOAD_DIR)

model.to(DEVICE)
model.eval()

print("✅ Model loaded")

# ================================
# BUILD FULL DATASET
# ================================
samples = []

for wav_path in Path("ravdess").rglob("*.wav"):
    parts = wav_path.stem.split("-")
    if len(parts) < 3:
        continue

    emotion_code = parts[2]
    label_str = EMOTION_MAP.get(emotion_code)
    if label_str is None:
        continue

    samples.append({
        "path": str(wav_path),
        "label": LABEL2ID[label_str]
    })

print(f"📊 Total dataset: {len(samples)}")

# ================================
# CREATE SMALL UNSEEN TEST SET
# ================================
random.seed(42)
random.shuffle(samples)

test_size = int(0.1 * len(samples))  # 10%
test_samples = samples[:test_size]

print(f"🧪 Test samples: {len(test_samples)}")

# ================================
# RUN PREDICTIONS
# ================================
all_preds = []
all_labels = []

for s in test_samples:
    wav, sr = torchaudio.load(s["path"])

    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    wav = wav.mean(dim=0)

    inputs = processor(
        wav.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(inputs.input_values.to(DEVICE)).logits

    pred = logits.argmax(dim=1).item()

    all_preds.append(pred)
    all_labels.append(s["label"])

print("✅ Predictions done")

# ================================
# METRICS
# ================================
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = (all_preds == all_labels).mean()

print("\n========== 🧪 SMALL TEST PERFORMANCE ==========")
print(f"Accuracy: {accuracy:.3f}")

print("\n========== 📊 CLASSIFICATION REPORT ==========")
print(classification_report(all_labels, all_preds, target_names=list(ID2LABEL.values())))

print("\n========== 📊 CONFUSION MATRIX ==========")
print(confusion_matrix(all_labels, all_preds))
'''

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

✅ Model loaded
📊 Total dataset: 1440
🧪 Test samples: 144
✅ Predictions done

========== 🧪 SMALL TEST PERFORMANCE ==========
Accuracy: 0.938

========== 📊 CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

     neutral       1.00      0.67      0.80         6
        calm       0.75      1.00      0.86        18
       happy       1.00      0.96      0.98        24
         sad       0.90      0.78      0.84        23
       angry       1.00      1.00      1.00        18
     fearful       1.00      0.94      0.97        17
   disgusted       0.95      1.00      0.98        21
   surprised       1.00      1.00      1.00        17

    accuracy                           0.94       144
   macro avg       0.95      0.92      0.93       144
weighted avg       0.95      0.94      0.94       144


========== 📊 CONFUSION MATRIX ==========
[[ 4  2  0  0  0  0  0  0]
 [ 0 18  0  0  0  0  0  0]
 [ 0  0 23  1  0  0  0  0]
 [ 0  4  0 18  0  0  1  0]
 [ 0  0  0  

In [ ]:
!pip install -q peft transformers torchaudio datasets accelerate bitsandbytes soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 2.7 MB/s eta 0:00:00


In [ ]:
def predict_logits(audio_path):
    wav, sr = torchaudio.load(audio_path)

    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    wav = wav.mean(dim=0)

    inputs = processor(
        wav.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(inputs.input_values.to(DEVICE)).logits

    return logits

In [ ]:
import torch.nn.functional as F

def emotion_similarity(ref_audio, gen_audio):
    ref_logits = predict_logits(ref_audio)
    gen_logits = predict_logits(gen_audio)

    return F.cosine_similarity(ref_logits, gen_logits).item()

In [ ]:
PITCH_RANGE = [-2, -1, 0, 1, 2]
SPEED_RANGE = [0.9, 1.0, 1.1]
GAIN_RANGE = [0.8, 1.0, 1.2, 1.5]

In [ ]:
import librosa
import soundfile as sf
import numpy as np

def apply_filter(y, sr, pitch, speed, gain):
    y_mod = librosa.effects.time_stretch(y, rate=speed)
    y_mod = librosa.effects.pitch_shift(y_mod, sr=sr, n_steps=pitch)
    y_mod = y_mod * gain

    y_mod = y_mod / np.max(np.abs(y_mod))
    return y_mod

In [ ]:
def optimize_emotion(input_path, ref_path):
    y, sr = librosa.load(input_path, sr=16000)

    best_score = -1
    best_audio = None

    for p in PITCH_RANGE:
        for s in SPEED_RANGE:
            for g in GAIN_RANGE:

                y_mod = apply_filter(y, sr, p, s, g)

                temp_path = "temp.wav"
                sf.write(temp_path, y_mod, sr)

                score = emotion_similarity(ref_path, temp_path)

                if score > best_score:
                    best_score = score
                    best_audio = y_mod

    return best_audio, best_score

In [ ]:
print(english_text)
print(REF_TEXT)

My name is Nikhil George and I live in Hyderabad Road
The quick brown fox jumps over the lazy dog near the riverbank every single morning.


In [ ]:
best_audio, score = optimize_emotion(
    "final_output.wav",
    "input_audio.wav"
)

In [ ]:
import soundfile as sf

sf.write("final_output_emotion.wav", best_audio, 16000)

In [ ]:
from IPython.display import Audio

print("🔊 Original:")
display(Audio("final_output.wav"))

print("🔊 Emotion-Optimized:")
display(Audio("final_output_emotion.wav"))

🔊 Original:


🔊 Emotion-Optimized:


In [ ]:
!pip install -q streamlit pyngrok torch torchaudio soundfile openai-whisper deep-translator qwen-tts transformers librosa resemblyzer jiwer sacrebleu numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import soundfile as sf
import whisper
from deep_translator import GoogleTranslator
from qwen_tts import Qwen3TTSModel  # ✅ Fixed import (TTS, not TT)
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
import torchaudio
import numpy as np
import tempfile
import os
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
from sacrebleu.metrics import CHRF
from jiwer import wer

# ---------- Cached models ----------
@st.cache_resource
def load_whisper():
    return whisper.load_model("large")   # or "medium" if low RAM

@st.cache_resource
def load_tts():
    return Qwen3TTSModel.from_pretrained(  # ✅ Fixed class name
        "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

@st.cache_resource
def load_emotion(model_path="/content/drive/MyDrive/emotion_model"):
    processor = Wav2Vec2Processor.from_pretrained(model_path)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(model_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    return processor, model, device

@st.cache_resource
def load_encoder():
    return VoiceEncoder()

# ---------- Emotion prediction ----------
def predict_emotion(audio_path, processor, model, device):
    wav, sr = torchaudio.load(audio_path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.mean(dim=0)
    inputs = processor(wav.numpy(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    pred_id = logits.argmax(dim=1).item()
    return model.config.id2label[pred_id]

# ---------- Voice similarity ----------
def compute_similarity(ref_path, clone_path, encoder):
    ref_wav = preprocess_wav(Path(ref_path))
    clone_wav = preprocess_wav(Path(clone_path))
    ref_emb = encoder.embed_utterance(ref_wav)
    clone_emb = encoder.embed_utterance(clone_wav)
    return np.dot(ref_emb, clone_emb) / (np.linalg.norm(ref_emb) * np.linalg.norm(clone_emb))

# ---------- Streamlit UI ----------
st.set_page_config(page_title="Voice Cloning + Emotion", layout="wide")
st.title("🎙️ Cross-Lingual Voice Cloning with Emotion Analysis")
st.markdown("Clone your voice to **Telugu / Malayalam / Hindi** speech, then hear it in **English**.")

col1, col2 = st.columns(2)
with col1:
    st.subheader("1. Reference (English)")
    ref_audio = st.file_uploader("Upload your English reference (9-15 sec)", type=["wav","mp3","m4a"])
    ref_text = st.text_area("Exact text spoken in the reference", value="The quick brown fox jumps over the lazy dog near the riverbank every single morning.")
with col2:
    st.subheader("2. Source Audio (Foreign Language)")
    src_lang = st.selectbox("Select language", options=["te","ml","hi"], format_func=lambda x: {"te":"Telugu","ml":"Malayalam","hi":"Hindi"}[x])
    src_audio = st.file_uploader(f"Upload {src_lang} audio", type=["wav","mp3","m4a"])

if st.button("🚀 Translate & Clone & Detect Emotion"):
    if not ref_audio or not src_audio or not ref_text.strip():
        st.error("Please upload both files and provide reference text.")
        st.stop()

    # Save uploaded files to temp paths
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        f.write(ref_audio.read())
        ref_path = f.name
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        f.write(src_audio.read())
        src_path = f.name

    # Load models
    with st.spinner("Loading Whisper..."):
        whisper_model = load_whisper()
    with st.spinner("Loading Qwen3-TTS (first time downloads ~4.5GB)..."):
        tts_model = load_tts()
    with st.spinner("Loading emotion model..."):
        emot_processor, emot_model, emot_device = load_emotion()
    with st.spinner("Loading voice encoder..."):
        encoder = load_encoder()

    # Transcribe
    st.subheader("📝 Transcription & Translation")
    with st.spinner(f"Transcribing {src_lang}..."):
        result = whisper_model.transcribe(src_path, language=src_lang, task="transcribe", temperature=0.0, best_of=5, beam_size=5)
        source_text = result["text"].strip()
    st.write(f"**Original ({src_lang})**: {source_text}")

    # Translate
    with st.spinner("Translating to English..."):
        english_text = GoogleTranslator(source=src_lang, target="en").translate(source_text)
    st.write(f"**English translation**: {english_text}")

    # Voice clone
    st.subheader("🎤 Cloned Output")
    with st.spinner("Synthesizing in your cloned voice..."):
        wavs, sr = tts_model.generate_voice_clone(text=english_text, ref_audio=ref_path, ref_text=ref_text, language="English")
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
            out_path = f.name
            sf.write(out_path, wavs[0], sr)
    st.audio(out_path, format="audio/wav")

    # Emotion
    st.subheader("😊 Emotion of Input Audio")
    emotion = predict_emotion(src_path, emot_processor, emot_model, emot_device)
    st.success(f"**Predicted emotion**: {emotion.upper()}")

    # Voice similarity
    st.subheader("🔊 Voice Similarity")
    cos_sim = compute_similarity(ref_path, out_path, encoder)
    st.metric("Cosine similarity", f"{cos_sim:.4f}")
    if cos_sim > 0.85:
        st.success("Excellent - very close")
    elif cos_sim > 0.70:
        st.info("Good - recognisable")
    elif cos_sim > 0.55:
        st.warning("Fair - some resemblance")
    else:
        st.error("Poor - voice drifted")

    # Optional translation quality
    st.subheader("📊 Translation Quality")
    back_translated = GoogleTranslator(source="en", target=src_lang).translate(english_text)
    st.write(f"**Back‑translated**: {back_translated}")
    chrf = CHRF()
    chrf_score = chrf.sentence_score(back_translated, [source_text])
    st.metric("ChrF score", f"{chrf_score.score:.1f}")
    try:
        wer_score = wer(source_text, back_translated)
        st.metric("Round‑trip WER", f"{wer_score:.3f}")
    except:
        pass

    # Cleanup
    os.unlink(ref_path); os.unlink(src_path); os.unlink(out_path)

st.caption("Built with Whisper, Qwen3‑TTS, and your trained emotion model.")

Writing app.py


In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

# Kill any previous tunnels
ngrok.kill()

# Set your authtoken (sign up at ngrok.com and get it from https://dashboard.ngrok.com/auth)
# Replace "YOUR_AUTH_TOKEN" with your actual token
NGROK_AUTH_TOKEN = "3CQ4ObFZsdFzhbHIhb9PTo8WqPs_3qUzx7SaCcks1HRHrEi2i"   # <---- CHANGE THIS
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create a tunnel to port 8501 (default Streamlit port)
public_url = ngrok.connect(addr="8501", proto="http")
print(f"Streamlit app will be available at: {public_url}")

# Run Streamlit in the background
import subprocess
import time
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(5)  # Give it a few seconds to start
print("Streamlit is running. Click the URL above to open the interface.")

Streamlit app will be available at: NgrokTunnel: "https://supremacy-nutlike-elk.ngrok-free.dev" -> "http://localhost:8501"
Streamlit is running. Click the URL above to open the interface.
